# Spaceship Titanic - Complete Analysis

Alright, let's dive into this Spaceship Titanic dataset. I'll go through everything from loading the data, inspecting it, cleaning it up, building a baseline model, and then tuning it for better performance. I'll add some comments along the way and print out intermediate results so we can see what's happening at each step.

In [ ]:
# First things first, let's import what we need
from pathlib import Path
import numpy as np
import pandas as pd

# For modeling
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print("Imports done!")

In [ ]:
# Now, let's load the data. I need to handle the path carefully.
base_dir = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
train_path = base_dir / 'Data' / 'train.csv'
test_path = base_dir / 'Data' / 'test.csv'
sample_submission_path = base_dir / 'Data' / 'sample_submission.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print('Loaded files from:', base_dir)
print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Sample submission shape:', sample_submission.shape)

In [ ]:
# Let's inspect the data to get a feel for it
def inspect_df(name, df, include_target=False):
    print('=' * 80)
    print(f'Dataset: {name}')
    print('=' * 80)
    print('Shape:', df.shape)
    print('Dtypes:')
    print(df.dtypes)
    print('Missing values per column:')
    print(df.isna().sum().sort_values(ascending=False))
    print('Duplicates:')
    print('- Duplicate rows:', df.duplicated().sum())
    if 'PassengerId' in df.columns:
        print('- Duplicate PassengerId:', df['PassengerId'].duplicated().sum())
    if include_target and 'Transported' in df.columns:
        print('Class balance (Transported):')
        counts = df['Transported'].value_counts(dropna=False)
        ratios = df['Transported'].value_counts(normalize=True, dropna=False)
        print(pd.concat([counts.rename('count'), ratios.rename('ratio')], axis=1))
    print()

# Run inspections
inspect_df('train.csv', train, include_target=True)
inspect_df('test.csv', test)
inspect_df('sample_submission.csv', sample_submission)

## Cleaning and Parsing the Data

Okay, so we've got some missing values and need to parse some columns like PassengerId, Cabin, and Name. Let's create a function to clean this up.

In [ ]:
# Enhanced cleaning function based on what we learned
def clean_and_parse(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Parse PassengerId
    pid_split = df['PassengerId'].str.split('_', n=1, expand=True)
    df['GroupId'] = pid_split[0]
    df['GroupMemberId'] = pd.to_numeric(pid_split[1], errors='coerce')
    
    # Parse Cabin
    cabin_split = df['Cabin'].fillna('Unknown/Unknown/Unknown').str.split('/', n=2, expand=True)
    df['CabinDeck'] = cabin_split[0]
    df['CabinNum'] = pd.to_numeric(cabin_split[1], errors='coerce')
    df['CabinSide'] = cabin_split[2]
    
    # Parse Name
    name_split = df['Name'].fillna('Unknown Unknown').str.split(' ', n=1, expand=True)
    df['FirstName'] = name_split[0]
    df['LastName'] = name_split[1].fillna('Unknown')
    
    # Handle booleans
    df['CryoSleep'] = df['CryoSleep'].astype('boolean').fillna(False).astype(bool)
    df['VIP'] = df['VIP'].astype('boolean').fillna(False).astype(bool)
    
    # Numeric columns
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spend_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df['Age'] = df['Age'].fillna(df['Age'].median())
    
    # Derived features
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)
    df['GroupSize'] = df.groupby('GroupId')['PassengerId'].transform('count')
    
    # Additional features for tuning
    df['LogTotalSpend'] = np.log1p(df['TotalSpend'])
    df['AgeBucket'] = pd.cut(
        df['Age'],
        bins=[-1, 12, 18, 25, 40, 60, 200],
        labels=['child', 'teen', 'young_adult', 'adult', 'mid_age', 'senior']
    )
    df['HomePlanet_Destination'] = (
        df['HomePlanet'].astype(str).replace('nan', 'Unknown') + '_' +
        df['Destination'].astype(str).replace('nan', 'Unknown')
    )
    df['FamilyNameSize'] = df.groupby('LastName')['PassengerId'].transform('count')
    
    # Normalize booleans for sklearn
    for c in ['CryoSleep', 'VIP']:
        df[c] = df[c].map({True: 'True', False: 'False'})
    
    if 'Transported' in df.columns:
        df['Transported'] = df['Transported'].astype(bool)
    
    return df

print("Cleaning function defined!")

In [ ]:
# Apply the cleaning
train_clean = clean_and_parse(train)
test_clean = clean_and_parse(test)

print('Train cleaned shape:', train_clean.shape)
print('Test cleaned shape:', test_clean.shape)
print('New columns added:', [c for c in train_clean.columns if c not in train.columns])
print('\nFirst few rows of cleaned train:')
print(train_clean.head())

## Baseline Model

Now let's build a simple baseline model to get started.

In [ ]:
# Prepare features and target
target_col = 'Transported'
drop_cols = ['PassengerId', 'Name', 'Cabin']

X = train_clean.drop(columns=[target_col]).drop(columns=drop_cols, errors='ignore')
y = train_clean[target_col].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# Convert bools to objects
for c in ['CryoSleep', 'VIP']:
    if c in X.columns:
        X[c] = X[c].astype(object)
    if c in X_test.columns:
        X_test[c] = X_test[c].astype(object)

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)
print('Feature columns:', list(X.columns))

In [ ]:
# Set up preprocessing
cat_cols = X.select_dtypes(include=['object', 'category', 'boolean']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print('Categorical columns:', cat_cols)
print('Numeric columns:', num_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols),
    ]
)

# Baseline model
model = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', model)
])

print("Pipeline set up!")

In [ ]:
# Evaluate the baseline
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_train, y_train)
val_preds = pipeline.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f'Validation accuracy: {val_acc:.4f}')

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print('CV scores:', np.round(cv_scores, 4))
print(f'CV mean: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

In [ ]:
# Train on full data and make submission
pipeline.fit(X, y)
test_preds = pipeline.predict(X_test).astype(bool)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': test_preds
})

submission_path = base_dir / 'Data' / 'submission.csv'
submission.to_csv(submission_path, index=False)

print(f'Saved submission to: {submission_path}')
print('First few predictions:')
print(submission.head())

## Model Tuning

Alright, baseline is done. Now let's tune it up with some hyperparameter search.

In [ ]:
# For tuning, we'll use the enhanced features
# We already have train_clean and test_clean with extra features

# Rebuild X and X_test with all features
X_full = train_clean.drop(columns=[target_col]).drop(columns=drop_cols, errors='ignore')
y_full = train_clean[target_col].astype(int)
X_test_full = test_clean.drop(columns=drop_cols, errors='ignore')

for c in ['CryoSleep', 'VIP']:
    if c in X_full.columns:
        X_full[c] = X_full[c].astype(object)
    if c in X_test_full.columns:
        X_test_full[c] = X_test_full[c].astype(object)

print('Full X shape:', X_full.shape)
print('Full X_test shape:', X_test_full.shape)

In [ ]:
# Update preprocessor for dense output (needed for HGB)
cat_cols_full = X_full.select_dtypes(include=['object', 'category', 'boolean']).columns.tolist()
num_cols_full = X_full.select_dtypes(include=['number']).columns.tolist()

preprocessor_tuned = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols_full),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols_full),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Tuned preprocessor ready!")

In [ ]:
# Quick benchmark
rf_base = Pipeline([
    ('preprocess', preprocessor_tuned),
    ('model', RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1))
])

hgb_base = Pipeline([
    ('preprocess', preprocessor_tuned),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

rf_scores = cross_val_score(rf_base, X_full, y_full, cv=cv, scoring='accuracy', n_jobs=-1)
hgb_scores = cross_val_score(hgb_base, X_full, y_full, cv=cv, scoring='accuracy', n_jobs=-1)

print('RF CV mean:', round(rf_scores.mean(), 5), '+/-', round(rf_scores.std(), 5))
print('HGB CV mean:', round(hgb_scores.mean(), 5), '+/-', round(hgb_scores.std(), 5))

In [ ]:
# RF tuning
rf_pipe = Pipeline([
    ('preprocess', preprocessor_tuned),
    ('model', RandomForestClassifier(random_state=42, n_jobs=4))
])

rf_param_dist = {
    'model__n_estimators': [300, 500, 700, 900],
    'model__max_depth': [None, 10, 15, 20, 30],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 4, 8],
    'model__max_features': ['sqrt', 'log2', None],
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=18,
    scoring='accuracy',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=1,
    refit=True
)

rf_search.fit(X_full, y_full)
print('RF best score:', round(rf_search.best_score_, 5))
print('RF best params:', rf_search.best_params_)

In [ ]:
# HGB tuning
hgb_pipe = Pipeline([
    ('preprocess', preprocessor_tuned),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

hgb_param_dist = {
    'model__learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
    'model__max_iter': [150, 250, 350, 500],
    'model__max_leaf_nodes': [15, 31, 63, 127],
    'model__min_samples_leaf': [10, 20, 30, 40],
    'model__l2_regularization': [0.0, 0.01, 0.1, 1.0],
}

hgb_search = RandomizedSearchCV(
    estimator=hgb_pipe,
    param_distributions=hgb_param_dist,
    n_iter=18,
    scoring='accuracy',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=4,
    refit=True
)

hgb_search.fit(X_full, y_full)
print('HGB best score:', round(hgb_search.best_score_, 5))
print('HGB best params:', hgb_search.best_params_)

In [ ]:
# Select the best model
best_name = 'RandomForest' if rf_search.best_score_ >= hgb_search.best_score_ else 'HistGradientBoosting'
best_model = rf_search.best_estimator_ if best_name == 'RandomForest' else hgb_search.best_estimator_
best_score = max(rf_search.best_score_, hgb_search.best_score_)

print('Selected model:', best_name)
print('Selected CV score:', round(best_score, 5))

# Final predictions
best_model.fit(X_full, y_full)
test_preds_tuned = best_model.predict(X_test_full).astype(bool)

submission_tuned = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': test_preds_tuned
})

out_path = base_dir / 'Data' / 'submission_tuned.csv'
submission_tuned.to_csv(out_path, index=False)
print(f'Saved tuned submission to: {out_path}')
print('First few tuned predictions:')
print(submission_tuned.head())